In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------
ROUNDS_PATH = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2026.csv")
ODDS_XLSX   = Path("/Users/joshmacbook/python_projects/OAD/Data/in Use/Odds_and_Results.xlsx")

CUTOFF_DATE = pd.Timestamp("2026-01-22")  # "before 1/22/2026" => strictly <
EVENT_ID = 2
YEAR = 2026

WINDOWS = (40, 24, 12)

# ------------------------------------------------------------
# 1) Load field (dg_id list) from Odds_and_Results.xlsx
#    NOTE: adjust sheet_name if your file uses a specific tab name
# ------------------------------------------------------------
odds = pd.read_excel(ODDS_XLSX)

# normalize types (common gotcha if excel stores as float)
odds["year"] = pd.to_numeric(odds.get("year"), errors="coerce")
odds["event_id"] = pd.to_numeric(odds.get("event_id"), errors="coerce")
odds["dg_id"] = pd.to_numeric(odds.get("dg_id"), errors="coerce")

field_ids = (
    odds.loc[(odds["year"] == YEAR) & (odds["event_id"] == EVENT_ID), "dg_id"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

print(f"Field size (unique dg_id): {len(field_ids)}")

# ------------------------------------------------------------
# 2) Load rounds and filter to: in-field + round_date < cutoff
# ------------------------------------------------------------
usecols = [
    "round_date",
    "dg_id",
    "player_name",
    "sg_total",
    "sg_putt",            # you asked for it; if missing, we'll handle below
    "birdies",
    "eagles_or_better",
    "round_score",
]

rounds = pd.read_csv(ROUNDS_PATH, usecols=lambda c: c in set(usecols))

# required columns check
required = {"round_date","dg_id","player_name","sg_total","birdies","eagles_or_better","round_score"}
missing_required = required - set(rounds.columns)
if missing_required:
    raise ValueError(f"combined_rounds file is missing required columns: {sorted(missing_required)}")

# sg_putt is optional (because your "needed columns" list didn’t include it)
has_putt = "sg_putt" in rounds.columns

rounds["round_date"] = pd.to_datetime(rounds["round_date"], errors="coerce")
rounds["dg_id"] = pd.to_numeric(rounds["dg_id"], errors="coerce").astype("Int64")

rounds = rounds.dropna(subset=["round_date","dg_id"]).copy()
rounds["dg_id"] = rounds["dg_id"].astype(int)

rounds = rounds.loc[
    (rounds["dg_id"].isin(field_ids)) &
    (rounds["round_date"] < CUTOFF_DATE)
].copy()

# combined birdies + eagles_or_better per round
rounds["b_eob"] = pd.to_numeric(rounds["birdies"], errors="coerce").fillna(0) + \
                  pd.to_numeric(rounds["eagles_or_better"], errors="coerce").fillna(0)

# Sort so "last N" = most recent N before cutoff
rounds = rounds.sort_values(["dg_id", "round_date"], ascending=[True, False])

print(f"Filtered rounds rows: {len(rounds):,}")

# ------------------------------------------------------------
# 3) Compute last-N window means for each player
# ------------------------------------------------------------
def window_means_for_player(g: pd.DataFrame, windows=(40,24,12)) -> pd.Series:
    out = {}

    # stabilize player_name: take most recent non-null
    pn = g["player_name"].dropna()
    out["player_name"] = pn.iloc[0] if len(pn) else None

    for w in windows:
        sub = g.head(w)
        out[f"n_rounds_L{w}"] = len(sub)

        out[f"sg_total_L{w}"] = pd.to_numeric(sub["sg_total"], errors="coerce").mean()
        out[f"b_eob_L{w}"]    = pd.to_numeric(sub["b_eob"], errors="coerce").mean()
        out[f"round_score_L{w}"] = pd.to_numeric(sub["round_score"], errors="coerce").mean()

        if has_putt:
            out[f"sg_putt_L{w}"] = pd.to_numeric(sub["sg_putt"], errors="coerce").mean()
        else:
            out[f"sg_putt_L{w}"] = np.nan

    return pd.Series(out)

summary = (
    rounds.groupby("dg_id", sort=False, as_index=True)
          .apply(window_means_for_player, windows=WINDOWS)
          .reset_index()
)

# ------------------------------------------------------------
# 4) Ensure every field player appears (even if 0 rounds pre-cutoff)
# ------------------------------------------------------------
field_df = pd.DataFrame({"dg_id": field_ids})
out = field_df.merge(summary, on="dg_id", how="left")

# Optional: order columns cleanly
col_order = ["dg_id", "player_name"]
for w in WINDOWS:
    col_order += [
        f"n_rounds_L{w}",
        f"sg_total_L{w}",
        f"b_eob_L{w}",
        f"sg_putt_L{w}",
        f"round_score_L{w}",
    ]
out = out[col_order]

out.sort_values("sg_total_L40", ascending=False, inplace=True, na_position="last")
out.reset_index(drop=True, inplace=True)

out.head(20)


Field size (unique dg_id): 156
Filtered rounds rows: 37,256


/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_23576/2106402628.py:52: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  rounds = pd.read_csv(ROUNDS_PATH, usecols=lambda c: c in set(usecols))
/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_23576/2106402628.py:110: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(window_means_for_player, windows=WINDOWS)


,dg_id,player_name,n_rounds_L40,sg_total_L40,b_eob_L40,sg_putt_L40,round_score_L40,n_rounds_L24,sg_total_L24,b_eob_L24,sg_putt_L24,round_score_L24,n_rounds_L12,sg_total_L12,b_eob_L12,sg_putt_L12,round_score_L12
0,18417,"Scheffler, Scottie",40,2.994050,5.225000,0.624725,67.525000,24,3.110625,5.875000,0.853542,66.666667,12,2.429083,6.333333,0.572750,66.916667
1,17646,"Fitzpatrick, Matthew",40,1.931775,4.975000,0.793353,68.125000,24,2.178542,5.541667,1.239000,67.958333,12,1.584917,5.333333,NaN,68.166667
2,24968,"Griffin, Ben",40,1.794350,4.600000,0.674406,67.925000,24,1.991583,4.916667,0.720688,67.708333,12,2.053417,5.250000,-0.003000,67.500000
3,10419,"Noren, Alex",40,1.707675,5.175000,1.180944,68.075000,24,1.378583,5.125000,1.171833,68.291667,12,1.417583,5.916667,1.905500,67.833333
4,23838,"Hojgaard, Rasmus",40,1.618600,5.050000,1.147214,68.600000,24,1.547125,5.250000,2.297000,68.250000,12,1.290833,5.083333,NaN,68.333333
5,14578,"Henley, Russell",40,1.399050,3.800000,0.135575,69.250000,24,1.409417,4.166667,0.229667,68.416667,12,1.443333,4.833333,0.248833,68.000000
6,23323,"MacIntyre, Robert",40,1.299625,5.125000,0.838385,68.175000,24,1.236500,5.500000,0.337900,68.333333,12,1.132583,5.500000,0.679125,68.250000
7,27194,"Hall, Harry",40,1.264125,4.625000,1.136419,68.600000,24,0.833542,4.333333,0.725067,68.916667,12,0.831583,3.916667,0.687250,69.416667
8,26649,"Thorbjornsen, Michael",40,1.261650,4.725000,0.044548,68.425000,24,1.527167,4.750000,-0.246733,68.083333,12,1.190833,4.583333,0.242571,67.583333
9,23950,"Aberg, Ludvig",40,1.252550,4.775000,0.289750,68.575000,24,1.222875,5.041667,0.388583,68.208333,12,1.406750,5.583333,NaN,68.333333


In [2]:
import pandas as pd
xls = pd.ExcelFile(ODDS_XLSX)
xls.sheet_names


['Sheet1']

In [3]:
# Excel
OUT_XLSX = Path("/Data/Archive/ae_2026_field_last40_24_12_pre_2026-01-22.xlsx")
out.to_excel(OUT_XLSX, index=False)
print("Saved:", OUT_XLSX)


Saved: /Users/joshmacbook/python_projects/OAD/Data/in Use/ae_2026_field_last40_24_12_pre_2026-01-22.xlsx


In [1]:
from Scripts.data_io import load_rounds

df = load_rounds()

print("shape:", df.shape)
print("\ncolumns:")
print(df.columns.tolist())

# sanity: show dtypes for the ones we care about
cols = [
    "tour","year","round_date","event_completed","event_id",
    "dg_id","player_name",
    "round_score",
    "sg_total","sg_t2g","sg_ott","sg_app","sg_arg","sg_putt",
    "birdies","eagles_or_better",
]
present = [c for c in cols if c in df.columns]
missing = [c for c in cols if c not in df.columns]
print("\npresent:", present)
print("missing:", missing)

print("\ndtypes (present only):")
print(df[present].dtypes)


shape: (311660, 37)

columns:
['tour', 'year', 'season', 'event_completed', 'event_name', 'event_id', 'player_name', 'dg_id', 'fin_text', 'round_num', 'course_name', 'course_num', 'course_par', 'start_hole', 'teetime', 'round_score', 'sg_putt', 'sg_arg', 'sg_app', 'sg_ott', 'sg_t2g', 'sg_total', 'driving_dist', 'driving_acc', 'gir', 'scrambling', 'prox_rgh', 'prox_fw', 'great_shots', 'poor_shots', 'eagles_or_better', 'birdies', 'pars', 'bogies', 'doubles_or_worse', 'finish_num', 'round_date']

present: ['tour', 'year', 'round_date', 'event_completed', 'event_id', 'dg_id', 'player_name', 'round_score', 'sg_total', 'sg_t2g', 'sg_ott', 'sg_app', 'sg_arg', 'sg_putt', 'birdies', 'eagles_or_better']
missing: []

dtypes (present only):
tour                        object
year                         int64
round_date          datetime64[ns]
event_completed     datetime64[ns]
event_id                     int64
dg_id                        int64
player_name                 object
round_score     

In [1]:
import pandas as pd
from pathlib import Path

# adjust if your raw file is named differently
RAW = Path("/Users/joshmacbook/python_projects/OAD/Data/Clean/Leagues/2026_small.csv")
OUT = Path("/Users/joshmacbook/python_projects/OAD/Data/Clean/Leagues/2026_small_normalized.csv")

from Scripts.OAD import load_schedule, normalize_league_df  # where it's defined/used

season = 2026
sched = load_schedule(season)

df_raw = pd.read_csv(RAW, low_memory=False)

df_norm = normalize_league_df(
    df_raw,
    sched=sched,
    season=season,
    league_id_value="main",
)

OUT.parent.mkdir(parents=True, exist_ok=True)
df_norm.to_csv(OUT, index=False)

print("Wrote:", OUT)
print("Columns:", df_norm.columns.tolist())
print("Missing event_id rows:", df_norm["event_id"].isna().sum() if "event_id" in df_norm.columns else "no event_id col")



2026-01-25 22:24:18.089 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.090 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.091 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.091 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.092 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.092 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.093 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-01-25 22:24:18.093 WARNING streamlit.runtime.caching.cache_d

COURSE_SENSITIVITY_PATH: /Users/joshmacbook/python_projects/OAD/Data/Clean/Processed/course_sensitivity_table.csv
  exists?: True


2026-01-25 22:24:18.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.289 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.290 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.290 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-25 22:24:18.290 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
/Users/joshmacbook/python_projects/OAD/Scripts/features.py:101: FutureWarning: DataFrameGroupBy.apply operated on the 

NameError: name 'latest_snapshot_upto' is not defined

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# ----------------------------
# Paths (match your config)
# ----------------------------
BASE_DATA_DIR = Path("/Users/joshmacbook/python_projects/OAD/Data")
COMBINED_ROUNDS_PATH = BASE_DATA_DIR / "in Use" / "combined_rounds_all_2017_2026.csv"
OAD_2026_PATH        = BASE_DATA_DIR / "in Use" / "OAD_2026.xlsx"

COURSE_FIT_OUT  = BASE_DATA_DIR / "in Use" / "course_fit_2026_dg_style_5attr.csv"
PLAYER_SKILL_OUT = BASE_DATA_DIR / "in Use" / "player_skill_5attr_hist_to_2026.csv"

# ----------------------------
# Config knobs
# ----------------------------
TARGET_SEASON = 2026

MAX_ROUNDS = 150
MIN_ROUNDS = 20
MIN_OBS_PER_COURSE = 80
SHRINK_K = 400.0

HALF_LIVES = {
    "driving_dist": 25,
    "driving_acc": 25,
    "sg_app": 60,
    "sg_arg": 60,
    "sg_putt": 120,
}
SKILL_MAP = {
    "driving_dist": "skill_dist",
    "driving_acc":  "skill_acc",
    "sg_app":       "skill_app",
    "sg_arg":       "skill_arg",
    "sg_putt":      "skill_putt",
}

# Choose course scaling:
# - "percentile": DG-ish (winsorized 5–95% then minmax), usually avoids hard 0/1
# - "player_minmax": exact minmax across courses (guarantees a 0 and 1 per column)
COURSE_SCALE = "percentile"   # <- change if you want


# ----------------------------
# Helpers
# ----------------------------
def scale_ecdf(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Map values to an empirical CDF in (0,1) using mid-ranks:
      ecdf = (rank - 0.5) / n
    This avoids exact 0 and 1 by construction.
    """
    out = df.copy()
    for c in cols:
        x = pd.to_numeric(out[c], errors="coerce")
        ok = x.notna()
        n = int(ok.sum())
        if n <= 1:
            out[c] = np.nan
            continue

        ranks = x[ok].rank(method="average")  # 1..n
        out.loc[ok, c] = (ranks - 0.5) / n   # (0,1)
        out.loc[~ok, c] = np.nan

    return out

def exp_decay_skill(vals: np.ndarray, half_life: float) -> float:
    n = len(vals)
    idx = np.arange(n)
    lam = np.log(2) / half_life
    w = np.exp(-lam * idx)
    w = w / w.sum()
    return float(np.sum(w * vals))

def compute_player_skills(hist: pd.DataFrame) -> pd.DataFrame:
    need = ["dg_id", "player_name", "event_completed"] + list(HALF_LIVES.keys())
    missing = [c for c in need if c not in hist.columns]
    if missing:
        raise ValueError(f"Missing required columns in hist for skills: {missing}")

    def _agg(g: pd.DataFrame) -> pd.Series:
        g = g.sort_values("event_completed", ascending=False).head(MAX_ROUNDS).copy()
        n = len(g)
        out = {"dg_id": int(g["dg_id"].iloc[0]), "player_name": g["player_name"].iloc[0], "n_rounds": n}

        if n < MIN_ROUNDS:
            for outc in SKILL_MAP.values():
                out[outc] = np.nan
            return pd.Series(out)

        for stat, hl in HALF_LIVES.items():
            v = pd.to_numeric(g[stat], errors="coerce").to_numpy()
            if np.all(~np.isfinite(v)):
                out[SKILL_MAP[stat]] = np.nan
            else:
                out[SKILL_MAP[stat]] = exp_decay_skill(v, hl)

        return pd.Series(out)

    skills = hist.groupby("dg_id", group_keys=False).apply(_agg).reset_index(drop=True)
    skill_cols = list(SKILL_MAP.values())
    skills = skills.dropna(subset=skill_cols).copy()
    return skills

def scale_minmax(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        x = pd.to_numeric(out[c], errors="coerce")
        lo = float(np.nanmin(x.to_numpy()))
        hi = float(np.nanmax(x.to_numpy()))
        if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
            out[c] = (x - lo) / (hi - lo)
        else:
            out[c] = np.nan
        out[c] = out[c].clip(0.0, 1.0)
    return out

def scale_percentile(df: pd.DataFrame, cols: list[str], lo_q: float = 0.05, hi_q: float = 0.95) -> pd.DataFrame:
    out = df.copy()
    for c in cols:
        x = pd.to_numeric(out[c], errors="coerce")
        lo = float(x.quantile(lo_q))
        hi = float(x.quantile(hi_q))
        if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
            xw = x.clip(lo, hi)
            out[c] = (xw - lo) / (hi - lo)
        else:
            out[c] = np.nan
        out[c] = out[c].clip(0.0, 1.0)
    return out


# ----------------------------
# 1) Load combined rounds
# ----------------------------
combined = pd.read_csv(COMBINED_ROUNDS_PATH)

combined["tour"] = combined["tour"].astype(str).str.upper()
combined["season"] = pd.to_numeric(combined["season"], errors="coerce")
combined["year"] = pd.to_numeric(combined["year"], errors="coerce")
combined["dg_id"] = pd.to_numeric(combined["dg_id"], errors="coerce").astype("Int64")
combined["event_completed"] = pd.to_datetime(combined["event_completed"], errors="coerce")

req_cols = ["sg_app","sg_arg","sg_putt","sg_total","driving_dist","driving_acc"]
for c in req_cols:
    if c not in combined.columns:
        raise ValueError(f"Missing {c} in combined rounds")
    combined[c] = pd.to_numeric(combined[c], errors="coerce")

# ----------------------------
# 2) Target courses from OAD schedule (2026)
# ----------------------------
sched = pd.read_excel(OAD_2026_PATH)

if "course_num" not in sched.columns:
    raise ValueError("OAD_2026.xlsx missing course_num column")

target_courses = (
    sched[["course_num", "course_name"]].copy()
    if "course_name" in sched.columns
    else sched[["course_num"]].copy()
)
target_courses["course_num"] = pd.to_numeric(target_courses["course_num"], errors="coerce")
target_courses = target_courses.dropna(subset=["course_num"]).drop_duplicates("course_num")
target_courses["course_num"] = target_courses["course_num"].astype(int)
target_courses = target_courses.sort_values("course_num").reset_index(drop=True)

print("Target courses:", len(target_courses))

# ----------------------------
# 3) Historical PGA rounds prior to target season
# ----------------------------
hist = combined[(combined["tour"] == "PGA") & (combined["season"] < TARGET_SEASON)].copy()

need_hist = ["dg_id","player_name","event_id","event_name","course_num","course_name","round_num","event_completed"] + req_cols
missing = [c for c in need_hist if c not in hist.columns]
if missing:
    raise ValueError(f"Missing columns in combined rounds needed for course fit: {missing}")

hist = hist.dropna(subset=need_hist).copy()
hist["course_num"] = pd.to_numeric(hist["course_num"], errors="coerce")
hist = hist.dropna(subset=["course_num"]).copy()
hist["course_num"] = hist["course_num"].astype(int)

hist = hist.sort_values(["dg_id","event_completed","event_id","round_num"]).reset_index(drop=True)

# ----------------------------
# 4) Player skills (exp-decay)
# ----------------------------
skills = compute_player_skills(hist)
print("Players with skills:", len(skills))

skill_cols = ["skill_dist","skill_acc","skill_app","skill_arg","skill_putt"]

# ----------------------------
# 5) Event-level perf merged with skills
# ----------------------------
event_perf = (
    hist.groupby(["event_id","event_name","course_num","course_name","dg_id"], as_index=False)
        .agg(event_sg_total=("sg_total","sum"))
)

event_perf = event_perf.merge(skills[["dg_id"] + skill_cols], on="dg_id", how="inner")
print("Event-player rows after merge:", len(event_perf))

event_perf["event_sg_total_centered"] = (
    event_perf["event_sg_total"]
    - event_perf.groupby(["event_id","course_num"])["event_sg_total"].transform("mean")
)

# z-scores globally for skills
z_map = {
    "skill_dist": "z_dist",
    "skill_acc":  "z_acc",
    "skill_app":  "z_app",
    "skill_arg":  "z_arg",
    "skill_putt": "z_putt",
}
for col, zname in z_map.items():
    mu = event_perf[col].mean()
    sd = event_perf[col].std(ddof=0)
    event_perf[zname] = 0.0 if (sd == 0 or np.isnan(sd)) else (event_perf[col] - mu) / sd

# ----------------------------
# 6) Course profiles -> pow_* -> imp_*
# ----------------------------
profiles = []
for _, crow in target_courses.iterrows():
    cnum = int(crow["course_num"])
    cname = str(crow["course_name"]) if "course_name" in crow.index else ""

    df_c = event_perf[event_perf["course_num"] == cnum].copy()
    if len(df_c) < MIN_OBS_PER_COURSE:
        continue

    df_c = df_c.dropna(subset=["event_sg_total_centered","z_dist","z_acc","z_app","z_arg","z_putt"]).copy()
    if len(df_c) < MIN_OBS_PER_COURSE:
        continue

    X = df_c[["z_dist","z_acc","z_app","z_arg","z_putt"]].to_numpy()
    y = df_c["event_sg_total_centered"].to_numpy()

    model = LinearRegression()
    model.fit(X, y)

    r2 = float(model.score(X, y)) if len(y) > 3 else 0.0
    r2 = float(np.clip(r2, 0.0, 1.0))
    predictability = float(np.sqrt(r2))

    abs_b = np.abs(model.coef_)
    pow_dist, pow_acc, pow_app, pow_arg, pow_putt = abs_b.tolist()

    # DG-ish: downweight noisy courses + shrink
    pow_dist *= predictability
    pow_acc  *= predictability
    pow_app  *= predictability
    pow_arg  *= predictability
    pow_putt *= predictability

    n_obs = int(len(df_c))
    shrink = float(n_obs / (n_obs + SHRINK_K))

    pow_dist *= shrink
    pow_acc  *= shrink
    pow_app  *= shrink
    pow_arg  *= shrink
    pow_putt *= shrink

    profiles.append({
        "course_num": cnum,
        "course_name": cname,
        "pow_dist": pow_dist,
        "pow_acc":  pow_acc,
        "pow_app":  pow_app,
        "pow_arg":  pow_arg,
        "pow_putt": pow_putt,
        "r2": r2,
        "n_obs": n_obs,
        "n_events": int(df_c["event_id"].nunique()),
        "n_players": int(df_c["dg_id"].nunique()),
        "target_season": TARGET_SEASON,
    })

course_profiles = pd.DataFrame(profiles).sort_values("course_num").reset_index(drop=True)
print("Profiles created:", len(course_profiles))

pow_cols = ["pow_dist","pow_acc","pow_app","pow_arg","pow_putt"]

course_profiles = scale_ecdf(course_profiles, pow_cols)

course_profiles = course_profiles.rename(columns={
    "pow_dist": "imp_dist",
    "pow_acc":  "imp_acc",
    "pow_app":  "imp_app",
    "pow_arg":  "imp_arg",
    "pow_putt": "imp_putt",
})

display(course_profiles.head(10))

# Quick sanity: do we still have tons of exact 0/1?
imp_cols = ["imp_dist","imp_acc","imp_app","imp_arg","imp_putt"]
print("\nExact 0/1 counts (by column):")
for c in imp_cols:
    s = pd.to_numeric(course_profiles[c], errors="coerce")
    print(c, "zeros:", int((s == 0).sum()), "ones:", int((s == 1).sum()), "n:", int(s.notna().sum()))

# ----------------------------
# 7) Save outputs
# ----------------------------
COURSE_FIT_OUT.parent.mkdir(parents=True, exist_ok=True)
PLAYER_SKILL_OUT.parent.mkdir(parents=True, exist_ok=True)

course_profiles.to_csv(COURSE_FIT_OUT, index=False)
skills.to_csv(PLAYER_SKILL_OUT, index=False)

print("\nSaved course profiles to:", COURSE_FIT_OUT)
print("Saved player skills to:  ", PLAYER_SKILL_OUT)


/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_55287/2767499569.py:140: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  combined = pd.read_csv(COMBINED_ROUNDS_PATH)


Target courses: 31
Players with skills: 570
Event-player rows after merge: 25541
Profiles created: 27


/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_55287/2767499569.py:104: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  skills = hist.groupby("dg_id", group_keys=False).apply(_agg).reset_index(drop=True)


,course_num,course_name,imp_dist,imp_acc,imp_app,imp_arg,imp_putt,r2,n_obs,n_events,n_players,target_season
0,4,Torrey Pines Golf Course (South Course),0.462963,0.018519,0.203704,0.314815,0.277778,0.115004,715,1,435,2026
1,5,Pebble Beach Golf Links,0.055556,0.166667,0.166667,0.203704,0.018519,0.084238,549,1,391,2026
2,6,Waialae Country Club,0.129630,0.240741,0.944444,0.796296,0.648148,0.175303,548,1,409,2026
3,9,Arnold Palmer's Bay Hill Club & Lodge,0.981481,0.648148,0.425926,0.907407,0.722222,0.184582,428,1,322,2026
4,11,TPC Sawgrass (THE PLAYERS Stadium Course),0.833333,0.870370,0.833333,0.944444,0.537037,0.193327,471,1,344,2026
5,12,Harbour Town Golf Links,0.574074,0.796296,0.462963,0.722222,0.314815,0.137053,451,1,323,2026
6,14,Augusta National Golf Club,0.537037,0.685185,0.574074,0.870370,0.425926,0.226045,291,2,166,2026
7,21,Colonial Country Club,0.425926,0.759259,0.500000,0.500000,0.870370,0.148197,559,1,359,2026
8,23,Muirfield Village Golf Club,0.500000,0.277778,0.685185,0.611111,0.388889,0.133151,546,1,303,2026
9,500,The Riviera Country Club,0.388889,0.055556,0.277778,0.277778,0.685185,0.132745,649,1,312,2026



Exact 0/1 counts (by column):
imp_dist zeros: 0 ones: 0 n: 27
imp_acc zeros: 0 ones: 0 n: 27
imp_app zeros: 0 ones: 0 n: 27
imp_arg zeros: 0 ones: 0 n: 27
imp_putt zeros: 0 ones: 0 n: 27

Saved course profiles to: /Users/joshmacbook/python_projects/OAD/Data/in Use/course_fit_2026_dg_style_5attr.csv
Saved player skills to:   /Users/joshmacbook/python_projects/OAD/Data/in Use/player_skill_5attr_hist_to_2026.csv


In [1]:
import pandas as pd
from datetime import datetime

# 1. Setup File Paths
round_data_path = '/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2026.csv'
odds_results_path = '/Users/joshmacbook/python_projects/OAD/Data/in Use/Odds_and_Results.xlsx'
today = datetime.now().strftime('%Y-%m-%d')

# 2. Load Data
# Load round data and ensure date is in datetime format
df_rounds = pd.read_csv(round_data_path)
df_rounds['round_date'] = pd.to_datetime(df_rounds['round_date'])

# Load this week's tournament field (Sheet name might need adjustment if not 'Sheet1')
df_field = pd.read_excel(odds_results_path)

# 3. Filter and Prepare
# Filter out rounds from today onwards (only look at history)
df_rounds = df_rounds[df_rounds['round_date'] < today].sort_values(['dg_id', 'round_date'])

# Define the Strokes Gained categories from the image
sg_categories = ['sg_ott', 'sg_app', 'sg_arg', 'sg_putt', 'sg_t2g', 'sg_total']

def get_rolling_stats(player_id, rounds_count):
    """Pulls the mean SG for the last N rounds for a specific player."""
    player_history = df_rounds[df_rounds['dg_id'] == player_id].tail(rounds_count)

    # If player has no rounds, return None
    if player_history.empty:
        return {f"{cat}_last{rounds_count}": None for cat in sg_categories}

    # Calculate means
    means = player_history[sg_categories].mean().to_dict()
    return {f"{k}_last{rounds_count}": v for k, v in means.items()}

# 4. Process the Field
print(f"Processing stats for {len(df_field)} players...")

final_data = []
for _, row in df_field.iterrows():
    p_id = row['dg_id']
    player_row = row.to_dict()

    # Pull for each window: 12, 24, 40
    for window in [12, 24, 40]:
        stats = get_rolling_stats(p_id, window)
        player_row.update(stats)

    final_data.append(player_row)

# 5. Save Results
df_output = pd.DataFrame(final_data)
df_output.to_csv('WMPO_2026_StrokesGained_Form.csv', index=False)
print("Finished! File saved as 'WMPO_2026_StrokesGained_Form.csv'")

/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_23997/1434166202.py:11: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rounds = pd.read_csv(round_data_path)


Processing stats for 12527 players...
Finished! File saved as 'WMPO_2026_StrokesGained_Form.csv'


In [3]:
import pandas as pd
from datetime import datetime

# 1. Setup File Paths
round_data_path = '/Users/joshmacbook/python_projects/OAD/Data/in Use/combined_rounds_all_2017_2026.csv'
odds_results_path = '/Users/joshmacbook/python_projects/OAD/Data/in Use/Odds_and_Results.xlsx'
output_path = '/Users/joshmacbook/python_projects/OAD/Data/in Use/WMPO_2026_StrokesGained_Analysis.csv'

# 2. Load and Filter the Field for this week only
# We filter by tournament name and year to avoid pulling every player in history
df_odds = pd.read_excel(odds_results_path)
df_field = df_odds[(df_odds['event_name'] == 'WM Phoenix Open') & (df_odds['year'] == 2026)].copy()

# 3. Load Round Data and Sort
df_rounds = pd.read_csv(round_data_path)
df_rounds['round_date'] = pd.to_datetime(df_rounds['round_date'])
# Sort chronologically so .tail() gets the LATEST rounds
df_rounds = df_rounds.sort_values(['dg_id', 'round_date'])

# 4. Filter Round Data for "History Only" (Before Feb 5, 2026)
# This prevents today's or tomorrow's live data from being counted in 'Last 12'
cutoff_date = '2026-02-04'
df_rounds = df_rounds[df_rounds['round_date'] <= cutoff_date]

# Define the categories from your image
sg_categories = ['sg_ott', 'sg_app', 'sg_arg', 'sg_putt', 'sg_t2g', 'sg_total']

def calculate_rolling_stats(player_id):
    """Calculates means for 12, 24, and 40 round windows for a player."""
    player_history = df_rounds[df_rounds['dg_id'] == player_id]
    stats = {}

    for n in [12, 24, 40]:
        recent_n = player_history.tail(n)
        if not recent_n.empty:
            means = recent_n[sg_categories].mean().to_dict()
            # Rename keys to include window size (e.g., sg_ott_l12)
            for cat, val in means.items():
                stats[f"{cat}_l{n}"] = round(val, 3)
            stats[f"rounds_actual_l{n}"] = len(recent_n)
        else:
            for cat in sg_categories:
                stats[f"{cat}_l{n}"] = None
            stats[f"rounds_actual_l{n}"] = 0
    return stats

# 5. Apply to the filtered 2026 field
print(f"Calculating rolling stats for {len(df_field)} players in the 2026 WMPO field...")
results_list = []

for idx, row in df_field.iterrows():
    player_stats = calculate_rolling_stats(row['dg_id'])
    # Merge original row data (odds/name) with the new SG stats
    combined_row = {**row.to_dict(), **player_stats}
    results_list.append(combined_row)

# 6. Save final file
df_final = pd.DataFrame(results_list)
df_final.to_csv(output_path, index=False)
print(f"File successfully created: {output_path}")

/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_23997/3408744.py:15: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rounds = pd.read_csv(round_data_path)


Calculating rolling stats for 120 players in the 2026 WMPO field...
File successfully created: /Users/joshmacbook/python_projects/OAD/Data/in Use/WMPO_2026_StrokesGained_Analysis.csv
